In [18]:
!pip install -q keras-tuner

In [20]:
!pip install -q keras-tuner

import pandas as pd
import numpy as np
import keras_tuner as kt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv("/content/diabetes.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def build_model(hp):

    model = Sequential()

    num_layers = hp.Int(
        name="num_layers",
        min_value=1,
        max_value=3
    )

    for i in range(num_layers):

        model.add(
            Dense(
                units=hp.Int(
                    name=f"units_{i}",
                    min_value=8,
                    max_value=128,
                    step=8
                ),
                activation="relu"
            )
        )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            values=[
                "adam",
                "sgd",
                "rmsprop"
            ]
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="my_dir",
    project_name="diabetes_tuning"
)

tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

best_hp = tuner.get_best_hyperparameters(1)[0]

print("\nBest Hyperparameters\n")

print("Number of Hidden Layers:", best_hp.get("num_layers"))

for i in range(best_hp.get("num_layers")):
    print(f"Units in Hidden Layer {i+1}:",
          best_hp.get(f"units_{i}"))

print("Best Optimizer:", best_hp.get("optimizer"))

best_model = tuner.get_best_models(1)[0]

loss, accuracy = best_model.evaluate(
    X_test,
    y_test
)

print("\nTest Accuracy:", accuracy)

Reloading Tuner from my_dir/diabetes_tuning/tuner0.json

Best Hyperparameters

Number of Hidden Layers: 1
Units in Hidden Layer 1: 96
Best Optimizer: adam


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8117 - loss: 0.4956  

Test Accuracy: 0.8116883039474487
